In [2]:
from scipy.ndimage import grey_opening
from scipy.stats import linregress
import numpy as np
import pandas as pd

In [ ]:
def lulu_baseline(intensity, burst_peak, window_hr, smooth_hr, cadence="10min"):

    signal = intensity.loc[
        burst_peak - pd.Timedelta(hours=window_hr/5):
        burst_peak + pd.Timedelta(hours=window_hr)
    ].copy()

    n = max(1, round(pd.Timedelta(hours=smooth_hr) / pd.Timedelta(cadence)))

    # Ensure regular cadence so missing timestamps are also detected
    signal = signal.asfreq(cadence)

    # Fill only short internal gaps (up to 3 samples)
    filled = signal.interpolate(method="time", limit=3, limit_area="inside")

    baseline = pd.Series(np.nan, index=filled.index)

    # Identify continuous segments without missing values
    valid = filled.notna()
    groups = valid.ne(valid.shift()).cumsum()

    for _, segment in filled.groupby(groups):

        if segment.isna().any():
            continue

        if len(segment) < n + 1:
            continue

        baseline.loc[segment.index] = grey_opening(segment.to_numpy(), size=n + 1)

    return baseline

In [ ]:
def shift_baseline(intensity, baseline_fit, burst_base, background_hr, quantile):

    signal = intensity.reindex(baseline_fit.index)
    background = signal.loc[burst_base - pd.Timedelta(hours=background_hr): burst_base]
    difference = (background - baseline_fit.reindex(background.index)).dropna()

    if difference.empty:
        return baseline_fit * np.nan, np.nan

    offset = difference.quantile(quantile)

    return baseline_fit + offset, offset

In [ ]:
def detect(threshold, signal, burst_peak, burst_base, cadence_td, rotation):
    for t in signal.loc[burst_peak:].index:
        if pd.isna(signal.loc[t]) or signal.loc[t] > threshold.loc[t]:
            continue

        duration = t - burst_base
        flag = int(duration < rotation)

        confirmation = rotation * (1 + flag)
        check_end = t + confirmation

        if check_end > signal.index[-1]:
            continue

        window = signal.loc[t:check_end]
        threshold_window = threshold.loc[t:check_end]

        expected = int(confirmation / cadence_td) + 1

        if len(window) < expected or window.isna().any():
            continue

        if (window > threshold_window).any():
            continue

        return t, flag

    return pd.NaT, 0

In [ ]:
def detect_fallback(threshold, signal, burst_peak, burst_base, cadence_td, rotation):

    longest_duration = pd.Timedelta(0)
    active_end = pd.NaT

    for t in signal.loc[burst_peak:].index:
        if pd.isna(signal.loc[t]) or signal.loc[t] > threshold.loc[t]:
            #print(" I am here")
            continue

        # Locating the longest continuous quiet interval
        win = signal.loc[t:]
        quiet = win.notna() & (win <=threshold.reindex(win.index))

        # Stopping at the onset of the identified quite interval
        end_idx =  np.flatnonzero(~quiet.to_numpy())
        n_quiet = end_idx[0] if len(end_idx) else len(quiet)

        duration = n_quiet * cadence_td
        if duration > longest_duration:
            longest_duration = duration
            active_end = t

    return active_end, 2

In [ ]:
def find_active_end_linear(intensity, baseline_fit, threshold_fit,  burst_peak, burst_base, rotation_hr, cadence):
    signal = intensity.reindex(baseline_fit.index)

    rotation = pd.Timedelta(hours=rotation_hr)
    cadence_td = pd.Timedelta(cadence)

    # PRIMARY: pre-event shifted threshold
    active_end, flag = detect(threshold_fit, signal, burst_peak, burst_base, cadence_td, rotation)
    threshold_method = 0
    threshold_used = threshold_fit

    # FALLBACK: estimate threshold from post-peak signal
    if pd.isna(active_end):

        after_peak = signal.loc[burst_peak:].dropna()
        difference = (after_peak - baseline_fit.reindex(after_peak.index)).dropna()

        if not difference.empty:
            # Lower-intensity population, excluding major bursts
            positive = difference[difference>0]

            if not positive.empty:
                offset = positive.quantile(0.75)
                threshold_fallback = baseline_fit + offset
                threshold_method = 1
                threshold_used= threshold_fallback
                #print("hello")
                active_end, flag = detect(threshold_fallback, signal, burst_peak, burst_base, cadence_td, rotation)

    if pd.isna(active_end):
        active_end, flag = detect_fallback(threshold_fit, signal, burst_peak, burst_base, cadence_td, rotation)

    return active_end, flag, threshold_method, threshold_used

In [ ]:
def find_active_start_linear(intensity, threshold_fit, prominent_base,
                             lookback_hr, rotation_hr, cadence="10min"):

    # Extend the fitted threshold backward
    slope = (threshold_fit.iloc[-1] - threshold_fit.iloc[0]) / (
        (threshold_fit.index[-1] - threshold_fit.index[0]).total_seconds() / 3600
    )

    index = pd.date_range(
        prominent_base - pd.Timedelta(hours=lookback_hr),
        prominent_base,
        freq=cadence
    )

    t = (index - threshold_fit.index[0]).total_seconds() / 3600
    threshold = pd.Series(threshold_fit.iloc[0] + slope*t, index=index)

    difference = threshold - intensity.reindex(index)

    n = int(np.ceil(rotation_hr * 3600 / pd.Timedelta(cadence).total_seconds()))
    count = 0

    # Search backward for one continuous quiet rotation
    for t, value in difference.iloc[::-1].items():

        if pd.notna(value) and value >= 0:
            count += 1
        else:
            count = 0

        if count >= n:
            active = difference.loc[t:prominent_base]
            active = active[active < 0]

            return (active.index[0], 0) if not active.empty else (prominent_base, 1)

    # No full quiet rotation found: onset is unresolved
    active = difference[difference < 0]

    if not active.empty:
        return active.index[0], 1

    return prominent_base, 1